Install and Import Required Libraries
Run this in your first cell to install any necessary scraping and machine learning libraries.

In [24]:
# Run this cell to install required libraries if not already installed
%pip install pandas scikit-learn skl2onnx onnxruntime openvino onnx requests joblib

Note: you may need to restart the kernel to use updated packages.



Download Directly in Your Notebook

Once the environment variables are injected, the official Python library automatically authenticates without requiring a kaggle.json file. Run this script directly inside your OpenShift AI notebook cell:

In [2]:
import os
import subprocess

# 1. Install the Kaggle CLI tool
subprocess.run(["pip", "install", "--quiet", "kaggle"])

# 2. Define the dataset slug (e.g., 50k IMDb reviews)
dataset_slug = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
download_path = "./data"

# 3. Execute the secure download and unzip
os.makedirs(download_path, exist_ok=True)
subprocess.run([
    "kaggle", "datasets", "download", 
    "-d", dataset_slug, 
    "-p", download_path, 
    "--unzip"
])

print(f"Dataset downloaded successfully to: {download_path}")


Dataset downloaded successfully to: ./data


Load Kaggle Dataset

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load the 50K IMDB dataset from Kaggle
# Make sure 'IMDB Dataset.csv' is in the same directory as your notebook
df_kaggle = pd.read_csv('./data/IMDB Dataset.csv')

# Convert text labels ('positive'/'negative') to binary (1/0)
df_kaggle['label'] = df_kaggle['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("Dataset shape:", df_kaggle.shape)
df_kaggle.head()

Dataset shape: (50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


Train the Sentiment Analysis Model
We train the model using the 50K dataset with TF-IDF vectorization and Logistic Regression.

In [22]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import onnx
from onnx import helper, numpy_helper, TensorProto
import openvino as ov
import os

modelpath = "./movie_review_sentiment_model/1"
onnxmodelpath = "./movie_review_sentiment_model_onnx/1"

os.makedirs(modelpath, exist_ok=True)
os.makedirs(onnxmodelpath, exist_ok=True)

# 1. Prepare data and train model
X = df_kaggle['review']
y = df_kaggle['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Vectorizer (10,000 features)
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)

# Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train) 

# Save the text vectorizer for inference preprocessing
joblib.dump(vectorizer, os.path.join(modelpath, "tfidf_vectorizer.pkl"))

# 2. Export to standard ONNX using dynamic feature sizing
pure_onnx_path = os.path.join(onnxmodelpath, "logistic_pure.onnx")

def export_lr_to_onnx(model, filename="logistic_pure.onnx"):
    coef = model.coef_
    intercept = model.intercept_
    classes = model.classes_
    
    # Dynamically extract the exact feature dimension from the trained weights
    n_features = coef.shape[1] 
    is_binary = (coef.shape[0] == 1 and len(classes) == 2)
    
    input_info = helper.make_tensor_value_info('float_input', TensorProto.FLOAT, [None, n_features])
    
    if is_binary:
        w_val = coef.T.astype(np.float32) # shape: [n_features, 1]
        b_val = intercept.astype(np.float32)
        
        weights_tensor = numpy_helper.from_array(w_val, name='weights')
        bias_tensor = numpy_helper.from_array(b_val, name='bias')
        zero_tensor = numpy_helper.from_array(np.array([0.0], dtype=np.float32), name='zero')
        
        nodes = [
            helper.make_node('MatMul', ['float_input', 'weights'], ['matmul_out']),
            helper.make_node('Add', ['matmul_out', 'bias'], ['decision']),
            helper.make_node('Greater', ['decision', 'zero'], ['greater_out']),
            helper.make_node('Cast', ['greater_out'], ['output_label'], to=TensorProto.INT64)
        ]
        output_info = helper.make_tensor_value_info('output_label', TensorProto.INT64, [None, 1])
        initializers = [weights_tensor, bias_tensor, zero_tensor]
    else:
        w_val = coef.T.astype(np.float32) # shape: [n_features, n_classes]
        b_val = intercept.astype(np.float32)
        
        weights_tensor = numpy_helper.from_array(w_val, name='weights')
        bias_tensor = numpy_helper.from_array(b_val, name='bias')
        
        nodes = [
            helper.make_node('MatMul', ['float_input', 'weights'], ['matmul_out']),
            helper.make_node('Add', ['matmul_out', 'bias'], ['logits']),
            helper.make_node('ArgMax', ['logits'], ['output_label'], axis=1, keepdims=0)
        ]
        output_info = helper.make_tensor_value_info('output_label', TensorProto.INT64, [None])
        initializers = [weights_tensor, bias_tensor]

    graph_def = helper.make_graph(
        nodes,
        'logistic_regression_graph',
        [input_info],
        [output_info],
        initializer=initializers
    )
    
    onnx_model = helper.make_model(graph_def, producer_name='sklearn2openvino')
    onnx_model.opset_import[0].version = 12
    onnx.save(onnx_model, filename)

export_lr_to_onnx(model, filename=pure_onnx_path)

# 3. Convert the clean ONNX model to OpenVINO IR format
ov_model = ov.convert_model(pure_onnx_path)
ov.save_model(ov_model, os.path.join(modelpath, "logistic_regression_openvino.xml"))

print("Model successfully converted and saved as OpenVINO IR!")

Model successfully converted and saved as OpenVINO IR!


Predict Movie Category (Good or Bad) Using an IMDb Review Comments
This function combines the scraper and your trained model to analyze an IMDb Review Comments and output an overall verdict.

In [23]:
import joblib
import numpy as np
import openvino as ov
import os

modelpath = "./movie_review_sentiment_model/1"

os.makedirs(modelpath, exist_ok=True)

# Load the saved vectorizer and OpenVINO model
vectorizer = joblib.load(os.path.join(modelpath, "tfidf_vectorizer.pkl"))

core = ov.Core()
compiled_model = core.compile_model(os.path.join(modelpath, "logistic_regression_openvino.xml"), "CPU")
output_node = compiled_model.output(0)

# New text reviews to classify
sample_reviews = [
    "This movie was fantastic and thrilling!",
    "Terrible waste of time, completely boring."
]

# Step 1: Transform text to TF-IDF dense array features
X_new_vec = vectorizer.transform(sample_reviews).toarray().astype(np.float32)

# Step 2: Run high-speed inference using OpenVINO
raw_preds = compiled_model([X_new_vec])[output_node]

# Define label mapping (Adjust if your 0/1 encoding is reversed)
label_map = {
    1: "Positive 😃",
    0: "Negative 😞"
}

# Step 3: Display results with human-readable formatting
print("--- Sentiment Analysis Results ---")
for review, pred in zip(sample_reviews, raw_preds):
    # Handle scalar or 1D array outputs depending on the shape
    pred_val = int(pred[0]) if isinstance(pred, (np.ndarray, list)) else int(pred)
    sentiment = label_map.get(pred_val, "Unknown")
    
    print(f"Review: \"{review}\"")
    print(f"Prediction: {sentiment}\n")

--- Sentiment Analysis Results ---
Review: "This movie was fantastic and thrilling!"
Prediction: Positive 😃

Review: "Terrible waste of time, completely boring."
Prediction: Negative 😞



docker run -d --rm -v E:\DataScience\IMDBReview\movie_review_sentiment_model:/models -p 9000:9000 -p 8000:8000 openvino/model_server:latest --model_path /models --model_name movie_review_sentiment_model --port 9000 --rest_port 8000

In [33]:
import io
import joblib
import json
import numpy as np
import requests
import tarfile

dockermodelspath = "/models/1/"
modelname = "movie_review_sentiment_model"
vectorizer_file = "tfidf_vectorizer.pkl"
container_id = "6cfb8d44e289af406297321ed83d635629da95953976ba0ae4fbc843860c127f"  # Replace with your actual container ID
file_path_inside_container = f"{dockermodelspath}{vectorizer_file}"  #

# 1. Fetch the vectorizer via HTTP URL instead of a local/mounted path
# Docker Engine API endpoint for downloading files/archives from a container
url = f"http://localhost:2375/containers/{container_id}/archive"

# Specify the path of the file inside the container
params = {
    "path": file_path_inside_container
}

print("Fetching vectorizer from Docker container API...")
response = requests.get(url, params=params)

if response.status_code == 200:
    # Docker returns a tarball stream, so we open it in memory using BytesIO
    with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:") as tar:
        # Get the list of files inside the tar archive
        member_names = tar.getnames()
        print(f"Extracted archive members: {member_names}")
        
        # Extract the file object directly into joblib
        file_member = tar.extractfile(member_names[0])
        vectorizer = joblib.load(file_member)
        
    print("Vectorizer successfully loaded from Docker container via API!")
else:
    print(f"Failed to fetch file. Error {response.status_code}: {response.text}")

# New raw text reviews to classify
sample_reviews = [
    "Terrible waste of time, completely boring.",
    "This movie was fantastic and thrilling!"
   
]

# Step 2: Transform text strings to dense TF-IDF numerical features
X_new_vec = vectorizer.transform(sample_reviews).toarray().astype(np.float32)

# Step 3: Format the payload for OpenVINO Model Server (OVMS) REST API
url = f"http://localhost:8000/v2/models/{modelname}/infer"

payload = {
    "inputs": [
        {
            "name": "float_input",
            "shape": list(X_new_vec.shape),
            "datatype": "FP32",
            "data": X_new_vec.tolist(),
        }
    ]
}

# Step 4: Send HTTP POST request to the inference endpoint
response = requests.post(url, json=payload)

if response.status_code == 200:
  result_data = response.json()
  raw_preds = result_data["outputs"][0]["data"]

  # Define human-readable label mapping
  label_map = {1: "Positive 😃", 0: "Negative 😞"}

  print("\n--- Sentiment Analysis Results from Docker ---")
  for review, pred in zip(sample_reviews, raw_preds):
    pred_val = int(pred)
    sentiment = label_map.get(pred_val, "Unknown")

    print(f'Review: "{review}"')
    print(f"Prediction: {sentiment}\n")
else:
  print(f"Inference Error {response.status_code}: {response.text}")

Fetching vectorizer from Docker container API...
Extracted archive members: ['tfidf_vectorizer.pkl']
Vectorizer successfully loaded from Docker container via API!

--- Sentiment Analysis Results from Docker ---
Review: "Terrible waste of time, completely boring."
Prediction: Negative 😞

Review: "This movie was fantastic and thrilling!"
Prediction: Positive 😃

